# Module 6 Homework

In this homework we'll put what we learned about Spark in practice.

For this homework we will be using the Yellow 2025-11 data from the official website:

```bash
wget -c https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
```

In [1]:
import pyspark
from pyspark.sql import SparkSession

## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

> To install PySpark follow this [guide](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/06-batch/setup/)

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('hw6_batch') \
    .getOrCreate()

### Answer 1

In [5]:
spark.version

'4.1.1'

## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- 25MB
- 75MB
- 100MB


### Answer 2

In [6]:
df = spark.read.option("header", "true").parquet("yellow_tripdata_2025-11.parquet")

In [7]:
df.repartition(4).write.parquet("yellow/2025/11/", mode="overwrite")

In [39]:
!du -sh yellow/2025/11/*.parquet

25M	yellow/2025/11/part-00000-b37b500b-eb76-4ed1-a260-e9aee07b28cc-c000.snappy.parquet
25M	yellow/2025/11/part-00001-b37b500b-eb76-4ed1-a260-e9aee07b28cc-c000.snappy.parquet
25M	yellow/2025/11/part-00002-b37b500b-eb76-4ed1-a260-e9aee07b28cc-c000.snappy.parquet
25M	yellow/2025/11/part-00003-b37b500b-eb76-4ed1-a260-e9aee07b28cc-c000.snappy.parquet


**Answer**: 25MB

## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610
- 102,340
- 162,604
- 225,768

### Answer 3

In [9]:
from pyspark.sql import functions as F

In [10]:
df.filter(
    df.tpep_pickup_datetime.cast("date") == F.lit("2025-11-15")
).count()

162604

**Answer**: 162604

## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

- 22.7
- 58.2
- 90.6
- 134.5

### Answer 4

In [11]:
import pyspark.sql.functions as F

In [12]:
df4 = df.withColumn(
    "trip_duration",
    (F.unix_timestamp(df.tpep_dropoff_datetime) - F.unix_timestamp(df.tpep_pickup_datetime)) / 3600
)

In [13]:
df4.select(F.max(df4.trip_duration)).show()

+------------------+
|max(trip_duration)|
+------------------+
| 90.64666666666666|
+------------------+



**Answer**: 90.6

## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- 4040
- 8080

### Answer 5

`4040`

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

- Governor's Island/Ellis Island/Liberty Island
- Arden Heights
- Rikers Island
- Jamaica Bay

### Answer 6

Get the data:

In [17]:
!wget -c -qq https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [19]:
df_zones = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")

#### Spark Dataframe API version

Relevant columns to join:

- `PULocationID`
- `LocationID`

In [22]:
df_full = df.join(
    df_zones,
    on=df.PULocationID == df_zones.LocationID,
    how="inner"
)

In [30]:
df_full.groupBy("Zone").count().orderBy("count", ascending=True).show(5)

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+
only showing top 5 rows


#### Spark SQL version with temp views

In [31]:
df.createOrReplaceTempView("yellow_taxis")
df_zones.createOrReplaceTempView("zones")

In [35]:
spark.sql("""
SELECT
    Zone,
    COUNT(1) AS count
FROM
    yellow_taxis AS yt
JOIN zones AS z
    ON yt.PULocationID = z.LocationID
GROUP BY
    Zone
ORDER BY
    count ASC
LIMIT
    5;
""").show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+



**Answer**: Governor's Island/Ellis Island/Liberty Island 